In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from data_pipeline import cleaning_data

In [ ]:
df=pd.read_csv(r"C:\Users\thien\code\AIMY\house-prices-advanced-regression-techniques\train.csv")
target_name='SalePrice'
drop_threshold=0.6

In [ ]:
df=df.drop(columns='Id')

In [ ]:
X,y=cleaning_data(df,target_name=target_name,drop_threshold=drop_threshold)

In [ ]:
X.isna().any(axis=0).sum()

In [ ]:
X.columns

In [ ]:
from pandasql import sqldf

analys_sql

In [ ]:
# ── Hàm tiện ích để chạy SQL ──────────────────────────────────────────────────
pysqldf = lambda q: sqldf(q, globals())

In [ ]:
# ==============================================================================
# CÂU 1 – Thống kê tổng quan (trung bình, min, max) theo chất lượng tổng thể
# ==============================================================================
# Mục đích : Xem giá nhà (log) biến động như thế nào theo từng mức OverallQual.
#            Đây là feature có tương quan mạnh nhất với SalePrice.
# Kết quả  : OverallQual càng cao → avg_price càng lớn, khoảng dao động càng rộng.
q1 = pysqldf("""
    SELECT
        OverallQual,
        COUNT(*)                         AS so_nha,
        ROUND(AVG(SalePrice), 4)         AS avg_log_price,
        ROUND(MIN(SalePrice), 4)         AS min_log_price,
        ROUND(MAX(SalePrice), 4)         AS max_log_price
    FROM df
    GROUP BY OverallQual
    ORDER BY OverallQual
""")
print("\n[Q1] Giá trung bình theo OverallQual:")
print(q1.to_string(index=False))


In [ ]:
# ==============================================================================
# CÂU 2 – Top 5 Neighborhood có giá nhà trung bình cao nhất
# ==============================================================================
# Mục đích : Tìm khu vực đắt đất nhất để định giá theo vùng.
# Kết quả  : Các neighborhood thuộc vùng cao cấp như NridgHt, StoneBr thường dẫn đầu.
q2 = pysqldf("""
    SELECT
        Neighborhood,
        COUNT(*)                         AS so_nha,
        ROUND(AVG(SalePrice), 4)         AS avg_log_price
    FROM df
    GROUP BY Neighborhood
    ORDER BY avg_log_price DESC
    LIMIT 5
""")
print("\n[Q2] Top 5 Neighborhood đắt nhất:")
print(q2.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 3 – Phân phối nhà theo số phòng ngủ (BedroomAbvGr) và loại nhà (BldgType)
# ==============================================================================
# Mục đích : Hiểu cơ cấu sản phẩm – loại nhà nào có nhiều phòng ngủ nhất.
# Kết quả  : Nhà 1 gia đình (1Fam) chiếm đa số, thường 3 phòng ngủ.
q3 = pysqldf("""
    SELECT
        BldgType,
        BedroomAbvGr,
        COUNT(*) AS so_nha
    FROM df
    GROUP BY BldgType, BedroomAbvGr
    ORDER BY BldgType, BedroomAbvGr
""")
print("\n[Q3] Phân phối nhà theo BldgType & BedroomAbvGr:")
print(q3.to_string(index=False))


In [ ]:
# ==============================================================================
# CÂU 4 – Giá trung bình theo chất lượng bếp (KitchenQual)
# ==============================================================================
# Mục đích : Bếp chất lượng cao có ảnh hưởng đến giá không?
# Kết quả  : Ex > Gd > TA > Fa – chất lượng bếp tỉ lệ thuận rõ ràng với giá.
q4 = pysqldf("""
    SELECT
        KitchenQual,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY KitchenQual
    ORDER BY avg_log_price DESC
""")
print("\n[Q4] Giá trung bình theo KitchenQual:")
print(q4.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 5 – Nhà có điều hòa (CentralAir) vs không có – chênh lệch giá bao nhiêu?
# ==============================================================================
# Mục đích : Lượng hóa giá trị của hệ thống điều hòa trung tâm.
# Kết quả  : Nhà có CentralAir=Y thường cao hơn đáng kể so với N.
q5 = pysqldf("""
    SELECT
        CentralAir,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price,
        ROUND(MAX(SalePrice) - MIN(SalePrice), 4) AS range_price
    FROM df
    GROUP BY CentralAir
""")
print("\n[Q5] Ảnh hưởng CentralAir đến giá:")
print(q5.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 6 – Giá trung bình theo số xe garage (GarageCars)
# ==============================================================================
# Mục đích : Garage lớn hơn thì giá có tăng tuyến tính không?
# Kết quả  : GarageCars = 3 thường cao nhất; = 0 thấp nhất.
q6 = pysqldf("""
    SELECT
        GarageCars,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY GarageCars
    ORDER BY GarageCars
""")
print("\n[Q6] Giá trung bình theo GarageCars:")
print(q6.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 7 – Xu hướng giá nhà theo năm bán (YrSold) và tháng bán (MoSold)
# ==============================================================================
# Mục đích : Kiểm tra tính thời vụ – tháng nào bán đắt nhất?
# Kết quả  : Thường mùa hè (tháng 5-7) giá cao hơn mùa đông.
q7 = pysqldf("""
    SELECT
        YrSold,
        MoSold,
        COUNT(*)                   AS so_giao_dich,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY YrSold, MoSold
    ORDER BY YrSold, MoSold
""")
print("\n[Q7] Xu hướng giá theo YrSold & MoSold (5 dòng đầu):")
print(q7.head(10).to_string(index=False))


In [ ]:
# ==============================================================================
# CÂU 8 – Tỉ lệ nhà được lát vỉa hè (PavedDrive) theo loại lô đất (LotConfig)
# ==============================================================================
# Mục đích : Lô đất góc phố hay trong ngõ thì hay có vỉa hè hơn không?
# Kết quả  : Lô Corner & CulDSac thường có PavedDrive=Y cao hơn.
q8 = pysqldf("""
    SELECT
        LotConfig,
        PavedDrive,
        COUNT(*) AS so_nha,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY LotConfig), 2) AS pct
    FROM df
    GROUP BY LotConfig, PavedDrive
    ORDER BY LotConfig, PavedDrive
""")
print("\n[Q8] Tỉ lệ PavedDrive theo LotConfig:")
print(q8.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 9 – Top 10 nhà có diện tích sống (GrLivArea) lớn nhất
# ==============================================================================
# Mục đích : Tìm những căn nhà rộng nhất và kiểm tra giá tương ứng.
# Kết quả  : GrLivArea lớn không nhất thiết là đắt nhất (outliers đã winsorize).
q9 = pysqldf("""
    SELECT
        rowid                      AS id,
        Neighborhood,
        HouseStyle,
        ROUND(GrLivArea, 0)        AS dien_tich_song,
        OverallQual,
        ROUND(SalePrice, 4)        AS log_price
    FROM df
    ORDER BY GrLivArea DESC
    LIMIT 10
""")
print("\n[Q9] Top 10 nhà có GrLivArea lớn nhất:")
print(q9.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 10 – Số nhà có lò sưởi (Fireplaces > 0) theo từng khu vực
# ==============================================================================
# Mục đích : Lò sưởi phổ biến ở khu vực nào? Liên quan đến khí hậu/đẳng cấp.
# Kết quả  : Khu cao cấp thường có nhiều nhà với lò sưởi hơn.
q10 = pysqldf("""
    SELECT
        Neighborhood,
        SUM(CASE WHEN Fireplaces > 0 THEN 1 ELSE 0 END) AS co_lo_suoi,
        SUM(CASE WHEN Fireplaces = 0 THEN 1 ELSE 0 END) AS khong_lo_suoi,
        COUNT(*)                                          AS tong
    FROM df
    GROUP BY Neighborhood
    ORDER BY co_lo_suoi DESC
    LIMIT 8
""")
print("\n[Q10] Số nhà có lò sưởi theo Neighborhood:")
print(q10.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 11 – Giá trung bình theo loại mái nhà (RoofStyle)
# ==============================================================================
# Mục đích : Kiểu mái có ảnh hưởng đến giá trị thẩm mỹ và giá không?
# Kết quả  : Mái Hip thường cao hơn mái Gable phổ thông.
q11 = pysqldf("""
    SELECT
        RoofStyle,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY RoofStyle
    ORDER BY avg_log_price DESC
""")
print("\n[Q11] Giá trung bình theo RoofStyle:")
print(q11.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 12 – Phân tích tầng hầm: tổng diện tích tầng hầm (TotalBsmtSF) theo BsmtQual
# ==============================================================================
# Mục đích : Chất lượng tầng hầm có tương quan với diện tích tầng hầm không?
# Kết quả  : BsmtQual=Ex thường có TotalBsmtSF lớn nhất.
q12 = pysqldf("""
    SELECT
        BsmtQual,
        COUNT(*)                       AS so_nha,
        ROUND(AVG(TotalBsmtSF), 2)     AS avg_bsmt_sf,
        ROUND(AVG(SalePrice), 4)       AS avg_log_price
    FROM df
    GROUP BY BsmtQual
    ORDER BY avg_log_price DESC
""")
print("\n[Q12] Tầng hầm theo BsmtQual:")
print(q12.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 13 – Nhà xây trước 1980 vs sau 1980: giá và chất lượng trung bình
# ==============================================================================
# Mục đích : Nhà cũ vs nhà mới khác nhau như thế nào về giá và chất lượng?
# Kết quả  : Nhà sau 1980 thường có OverallQual và giá cao hơn.
q13 = pysqldf("""
    SELECT
        CASE WHEN YearBuilt < 1980 THEN 'Truoc 1980' ELSE 'Tu 1980 tro di' END AS thoi_ky,
        COUNT(*)                     AS so_nha,
        ROUND(AVG(OverallQual), 2)   AS avg_qual,
        ROUND(AVG(SalePrice), 4)     AS avg_log_price
    FROM df
    GROUP BY thoi_ky
""")
print("\n[Q13] Nhà cũ vs nhà mới:")
print(q13.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 14 – Diện tích boong gỗ (WoodDeckSF) vs hiên thông thoáng (OpenPorchSF): nhà nào có cả hai?
# ==============================================================================
# Mục đích : Tìm các nhà "outdoor friendly" – có cả deck lẫn porch.
# Kết quả  : Những nhà này thường có giá và diện tích sống cao hơn mức trung bình.
q14 = pysqldf("""
    SELECT
        CASE
            WHEN WoodDeckSF > 0 AND OpenPorchSF > 0 THEN 'Co ca hai'
            WHEN WoodDeckSF > 0                     THEN 'Chi co Deck'
            WHEN OpenPorchSF > 0                    THEN 'Chi co Porch'
            ELSE 'Khong co gi'
        END AS loai_ngoai_troi,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(GrLivArea), 2)   AS avg_dien_tich_song,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY loai_ngoai_troi
    ORDER BY avg_log_price DESC
""")
print("\n[Q14] Phân tích WoodDeck & OpenPorch:")
print(q14.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 15 – Phân tích loại hình bán hàng (SaleType) và điều kiện bán (SaleCondition)
# ==============================================================================
# Mục đích : Giao dịch "Normal" so với "Partial" (nhà mới chưa hoàn thiện) khác giá bao nhiêu?
# Kết quả  : SaleCondition=Partial thường cao hơn do nhà mới (New build).
q15 = pysqldf("""
    SELECT
        SaleType,
        SaleCondition,
        COUNT(*)                   AS so_giao_dich,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY SaleType, SaleCondition
    ORDER BY avg_log_price DESC
    LIMIT 10
""")
print("\n[Q15] Giá theo SaleType & SaleCondition:")
print(q15.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 16 – Nhà có pool (PoolArea > 0): đặc điểm và giá
# ==============================================================================
# Mục đích : Bể bơi đóng góp bao nhiêu vào giá trị căn nhà?
# Kết quả  : Nhà có bể bơi rất ít (~1%) nhưng giá trung bình cao hơn đáng kể.
q16 = pysqldf("""
    SELECT
        CASE WHEN PoolArea > 0 THEN 'Co be boi' ELSE 'Khong co' END AS co_be_boi,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(PoolArea), 2)    AS avg_pool_area,
        ROUND(AVG(OverallQual), 2) AS avg_qual,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY co_be_boi
""")
print("\n[Q16] Nhà có bể bơi vs không:")
print(q16.to_string(index=False))


In [ ]:
# ==============================================================================
# CÂU 17 – Số phòng tắm đầy đủ (FullBath + BsmtFullBath) và tác động lên giá
# ==============================================================================
# Mục đích : Tổng số phòng tắm (kể cả tầng hầm) ảnh hưởng thế nào đến giá?
# Kết quả  : Nhà có 3+ phòng tắm giá tăng rõ rệt.
q17 = pysqldf("""
    SELECT
        (FullBath + BsmtFullBath)  AS tong_phong_tam,
        COUNT(*)                   AS so_nha,
        ROUND(AVG(GrLivArea), 2)   AS avg_dien_tich_song,
        ROUND(AVG(SalePrice), 4)   AS avg_log_price
    FROM df
    GROUP BY tong_phong_tam
    ORDER BY tong_phong_tam
""")
print("\n[Q17] Giá theo tổng số phòng tắm đầy đủ:")
print(q17.to_string(index=False))


In [ ]:
# ==============================================================================
# CÂU 18 – Nhà được cải tạo (YearRemodAdd > YearBuilt) vs chưa cải tạo
# ==============================================================================
# Mục đích : Cải tạo có làm tăng giá so với nhà nguyên gốc không?
# Kết quả  : Nhà đã cải tạo thường có OverallCond và giá cao hơn.
q18 = pysqldf("""
    SELECT
        CASE WHEN YearRemodAdd > YearBuilt THEN 'Da cai tao' ELSE 'Chua cai tao' END AS tinh_trang,
        COUNT(*)                     AS so_nha,
        ROUND(AVG(OverallCond), 2)   AS avg_cond,
        ROUND(AVG(SalePrice), 4)     AS avg_log_price
    FROM df
    GROUP BY tinh_trang
""")
print("\n[Q18] Nhà đã cải tạo vs chưa cải tạo:")
print(q18.to_string(index=False))


In [ ]:
# ==============================================================================
# CÂU 19 – Phân vị giá (quartile) theo kiểu nhà (HouseStyle)
# ==============================================================================
# Mục đích : Phân tích phân phối giá trong từng kiểu nhà (1 tầng, 2 tầng...).
# Kết quả  : Nhà 2 tầng (2Story) thường có median và Q3 cao hơn nhà 1 tầng.
q19 = pysqldf("""
    SELECT
        HouseStyle,
        COUNT(*)                          AS so_nha,
        ROUND(MIN(SalePrice), 4)          AS min_price,
        ROUND(AVG(SalePrice), 4)          AS avg_price,
        ROUND(MAX(SalePrice), 4)          AS max_price
    FROM df
    GROUP BY HouseStyle
    ORDER BY avg_price DESC
""")
print("\n[Q19] Phân vị giá theo HouseStyle:")
print(q19.to_string(index=False))

In [ ]:
# ==============================================================================
# CÂU 20 – Kết hợp nhiều điều kiện: nhà "đáng mua" (chất lượng cao, giá hợp lý)
# ==============================================================================
# Mục đích : Xác định nhà có OverallQual >= 7, GrLivArea >= 1500,
#            GarageCars >= 2, CentralAir = 'Y' → "nhà đáng mua".
# Kết quả  : Lọc ra những căn nhà tốt nhất về mặt feature để phân tích thêm.
q20 = pysqldf("""
    SELECT
        rowid                      AS id,
        Neighborhood,
        YearBuilt,
        OverallQual,
        ROUND(GrLivArea, 0)        AS dien_tich_song,
        GarageCars,
        CentralAir,
        KitchenQual,
        ROUND(SalePrice, 4)        AS log_price
    FROM df
    WHERE OverallQual >= 7
      AND GrLivArea  >= 1500
      AND GarageCars >= 2
      AND CentralAir = 'Y'
    ORDER BY SalePrice DESC
    LIMIT 15
""")
print("\n[Q20] Danh sách nhà 'đáng mua' (chất lượng cao, đủ tiện nghi):")
print(q20.to_string(index=False))

print("\n" + "="*80)
print("Hoàn thành 20 câu SQL phân tích dữ liệu House Prices!")
print("="*80)